In [1]:
!pip install pandas scikit-learn joblib

In [2]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import joblib

In [3]:
from google.colab import files
uploaded = files.upload()


Saving Final_NGO_Milestone_Dataset.csv to Final_NGO_Milestone_Dataset.csv


In [4]:
df=pd.read_csv('/content/Final_NGO_Milestone_Dataset.csv', encoding='latin1')

In [5]:
df.head()

,NGO_Name,Timestamp,Receipts_Uploaded,Milestone_1,Exp_1,Req_1,Milestone_2,Exp_2,Req_2,Milestone_3,Exp_3,Req_3
0,Smile Foundation,6/1/2025 10:00,1,Food Distribution,20000,22000,Transport,15000,15000,Medical Kit,10000,12000
1,Goonj,6/2/2025 11:30,0,Transport,15000,19000,Medical Kit,10000,10000,NaN,0,0
2,Akshaya Patra Foundation,6/3/2025 9:45,1,Food Distribution,25000,25000,NaN,0,0,NaN,0,0
3,HelpAge India,6/4/2025 12:20,1,Medical Kit,20000,22000,Transport,10000,10000,NaN,0,0
4,Teach For India,6/5/2025 14:00,0,Education Support,20000,26000,Healthcare Camp,15000,15000,NaN,0,0


In [6]:
# df['Deviation_1'] = df['Req_1'] - df['Exp_1']
# df['Deviation_2'] = df['Req_2'] - df['Exp_2']
# df['Deviation_3'] = df['Req_3'] - df['Exp_3']
# df['Total_Deviation'] = df[['Deviation_1', 'Deviation_2', 'Deviation_3']].sum(axis=1, skipna=True)



# Convert values to numeric
for col in ['Req_1', 'Exp_1', 'Req_2', 'Exp_2', 'Req_3', 'Exp_3']:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

# Feature engineering
df['Deviation_1'] = df['Req_1'] - df['Exp_1']
df['Deviation_2'] = df['Req_2'] - df['Exp_2']
df['Deviation_3'] = df['Req_3'] - df['Exp_3']

df['Total_Requested'] = df['Req_1'] + df['Req_2'] + df['Req_3']
df['Total_Used'] = df['Exp_1'] + df['Exp_2'] + df['Exp_3']
df['Total_Deviation'] = df['Total_Requested'] - df['Total_Used']

# Percent deviation
df['Deviation_Percent'] = (df['Total_Deviation'] / (df['Total_Requested'] + 1)) * 100

# Score components
df['score'] = 0

# High mismatch gets high score
df.loc[df['Deviation_Percent'] > 50, 'score'] += 2
df.loc[df['Deviation_Percent'] > 80, 'score'] += 2

# If any phase deviation is huge
df.loc[(df['Deviation_1'] > 1000) | (df['Deviation_2'] > 1000) | (df['Deviation_3'] > 1000), 'score'] += 2

# No receipts uploaded = suspicious
df.loc[df['Receipts_Uploaded'] == 0, 'score'] += 2

# Non-linear request growth across phases (possibly inflated)
df['req_diff_1_2'] = df['Req_2'] - df['Req_1']
df['req_diff_2_3'] = df['Req_3'] - df['Req_2']
df.loc[(df['req_diff_1_2'] > 2000) | (df['req_diff_2_3'] > 2000), 'score'] += 1

# Final fraud label based on total score
df['is_fraud'] = df['score'].apply(lambda x: 1 if x >= 4 else 0)


In [ ]:
# #  Step 5: Create Synthetic Labels for Fraud

# df['Is_Fraud'] = df['Total_Deviation'].apply(lambda x: 1 if x > 5000 else 0)




In [7]:
#  Step 6: Train/Test Split
# features = ['Deviation_1', 'Deviation_2', 'Deviation_3', 'Total_Deviation', 'Receipts_Uploaded']
# X = df[features].fillna(0)
# y = df['Is_Fraud']

# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)


from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
import joblib

features = ['Req_1', 'Exp_1', 'Deviation_1',
            'Req_2', 'Exp_2', 'Deviation_2',
            'Req_3', 'Exp_3', 'Deviation_3',
            'Receipts_Uploaded']

X = df[features]
y = df['is_fraud']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

clf = RandomForestClassifier(class_weight='balanced', random_state=42)
clf.fit(X_train, y_train)

joblib.dump(clf, 'fraud_rf_model.pkl')


['fraud_rf_model.pkl']

In [8]:
files.download("fraud_rf_model.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [9]:
from sklearn.metrics import classification_report

y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred))


              precision    recall  f1-score   support

           0       0.75      1.00      0.86         3
           1       1.00      0.67      0.80         3

    accuracy                           0.83         6
   macro avg       0.88      0.83      0.83         6
weighted avg       0.88      0.83      0.83         6



In [ ]:
# #  Step 7: Train the Model
# model = RandomForestClassifier(random_state=42)
# model.fit(X_train, y_train)

In [ ]:
# # Step 8: Evaluate the Model
# y_pred = model.predict(X_test)
# print("Classification Report:\n", classification_report(y_test, y_pred))

In [ ]:
# #  Step 9: Save the Model
# joblib.dump(model, "fraud_model.pkl")

In [ ]:
# #  Step 10: Download the Model File
# files.download("fraud_model.pkl")